In [1]:
import os
import re
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from dotenv import load_dotenv

load_dotenv(override=True)
model = ChatOpenAI(model="gpt-5.4-mini")

INTERVIEW_SYSTEM = """당신은 사용자가 만들고 싶은 'SKILL.md' 명세를 끌어내는 인터뷰어입니다.

목표: 사용자가 말한 스킬 아이디어에서 다음 항목들이 모두 명확해질 때까지 한 번에 1~3개씩 핵심 질문을 던지세요.
필수 확인 항목:
1. 스킬의 이름과 한 줄 요약 (description)
2. 어떤 상황/요청에서 이 스킬을 발동해야 하는가 (트리거)
3. 단계별 절차 (1단계, 2단계 …)
4. 각 단계에서 LLM이 절대 놓치면 안 되는 체크 항목
5. 결과를 어떤 형식/말투로 출력해야 하는가
6. 예외 상황 처리 방법

규칙:
- 사용자의 답이 모호하면 더 구체적으로 파고드세요. ("그 '검토'가 정확히 뭘 보는 건가요?")
- 이미 답한 내용은 다시 묻지 마세요.
- 사용자의 분야/맥락을 추측하지 말고 직접 물어보세요.
- 한국어로 답하세요.
- **응답 마지막에는 반드시 다음 문장을 그대로 출력하세요:**
  지금까지의 대화내역으로 SKILL을 만들고자 하시면 '만족'이라고 입력하세요.
"""

GENERATE_PROMPT = """지금까지의 인터뷰 대화를 바탕으로 완성된 SKILL.md 파일 본문을 생성하세요.

반드시 아래 형식을 그대로 따르세요. 코드펜스(```)나 추가 설명 없이 SKILL.md 본문만 출력하세요.

---
name: <소문자-하이픈-식별자>
description: <한 문장으로 이 스킬이 언제 쓰이는지>
---

# <스킬 제목>

## 사용 시기
- ...

## 절차
### 1단계. ...
- ...

### 2단계. ...
- ...

(필요한 만큼 단계를 추가)

## 출력 형식
...

## 예외 처리
- ...
"""

# 인터뷰 루프
history = [SystemMessage(content=INTERVIEW_SYSTEM)]

print("=" * 60)
print("[SKILL 만들기 인터뷰] 만들고 싶은 스킬에 대해 자유롭게 말씀해주세요.")
print("(종료하려면 '만족' 입력 / 강제 중단은 Ctrl+C)")
print("=" * 60)

while True:
    user_input = input("\n나> ").strip()
    if not user_input:
        continue

    if user_input == "만족":
        print("\n[인터뷰 종료] SKILL.md 생성 중...")
        break

    history.append(HumanMessage(content=user_input))
    ai_msg = model.invoke(history)
    history.append(AIMessage(content=ai_msg.content))
    print(f"\nLLM> {ai_msg.content}")

# SKILL.md 생성
gen_messages = history + [HumanMessage(content=GENERATE_PROMPT)]
skill_md = model.invoke(gen_messages).content.strip()

# 코드펜스가 섞여 들어와도 제거
skill_md = re.sub(r"^```(?:markdown|md)?\s*|\s*```$", "", skill_md, flags=re.MULTILINE).strip()

# name 추출해서 폴더명으로 사용
match = re.search(r"^name:\s*(\S+)", skill_md, re.MULTILINE)
skill_name = match.group(1) if match else "my-skill"

skill_dir = f"./skills/{skill_name}"
os.makedirs(skill_dir, exist_ok=True)
skill_path = f"{skill_dir}/SKILL.md"

with open(skill_path, "w", encoding="utf-8") as f:
    f.write(skill_md)

print("\n" + "=" * 60)
print(f"SKILL.md 생성 완료: {skill_path}")
print("=" * 60)
print(skill_md)


[SKILL 만들기 인터뷰] 만들고 싶은 스킬에 대해 자유롭게 말씀해주세요.
(종료하려면 '만족' 입력 / 강제 중단은 Ctrl+C)

LLM> 좋아요. 이 스킬은 “매일 5개 분야의 트렌드를 간단 요약해서 받는 자동 브리핑”으로 이해했어요.  
명세를 정확히 만들기 위해 몇 가지만 더 물어볼게요.

1) **발동 트리거**는 어떻게 할까요?  
- 사용자가 “오늘 트렌드 알려줘”처럼 요청할 때마다 실행할지,  
- 아니면 **매일 특정 시간에 자동으로** 생성하는지요?

2) **각 주제별 2개 트렌드의 형식**은 어떻게 원하시나요?  
- 예: 제목 + 한 줄 요약  
- 혹은 제목 + 2~3문장 요약 + 왜 중요한지  
- 또는 아주 짧게 핵심만 bullet로

3) **fortune teller**는 정확히 어떤 의미로 다룰까요?  
- 운세/사주/별자리/타로 같은 **예측형 콘텐츠**인지,  
- 아니면 “오늘의 기분/흐름” 같은 **가벼운 위트형 코너**인지 정해주시면 좋아요.

지금까지의 대화내역으로 SKILL을 만들고자 하시면 '만족'이라고 입력하세요.

LLM> 좋아요. 핵심은 꽤 선명해졌습니다.  
이제 SKILL.md에 넣을 **실행 절차와 예외 처리**를 마무리하려고 해요. 아래 3가지만 더 알려주세요.

1) **매일 아침 7시 브리핑의 구성 순서**를 어떻게 할까요?  
- 예: 정치 → 경제 → 사회 → 연예 → fortune teller 고정 순서  
- 또는 중요도/이슈도에 따라 재정렬  
- 혹은 분야별로 항상 동일하게 2개씩만 제공

2) **“왜 중요한지”**는 어느 정도 깊이로 써야 하나요?  
- 단순한 의미 설명인지  
- 아니면 경제/정치처럼 일부 항목은 **주식, 산업, 정책 영향**까지 연결해서 적어야 하는지  
- 그리고 이 연결은 “가능성” 수준으로만 말하면 되는지, 아니면 **구체적 종목/섹터 힌트**까지 제시해도 되는지요?

3) **예외 상황 처리**를 정해 주세요. 예를 들어  
- 해당 날짜에 이슈가 